In [1]:
#as demonstrated in documentation, a decision tree is esentially only a collection of linked nodes.
#to build this tree we must define its most basic bulding block, the Node.


#We need a class that can handle two different roles: decision nodes and leaf nodes.
#Decision nodes: These act like a fork in the road, where we decide which path to take based on a feature and a threshold.
#Leaf nodes: These are the endpoints of our tree, where we make a final prediction.

class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        # 1. Attributes for Decision Nodes
        self.feature = feature      #The index of the feature we are testing (e.g., Column 0)
        self.threshold = threshold  #The value we split on (e.g., is X <= 5?)
        self.left = left            #The "Yes/True" branch
        self.right = right          #The "No/False" branch

        # 2. Attribute for Leaf Nodes
        self.value = value          #The predicted class or mean value
    
    def is_leaf_node(self):
        """Helper to check if we have reached a final prediction."""
        return self.value is not None


By implementing this class:
- The trees become recursive by nature. By giving each node a left and right attribute that points to other Node objects, we can build a structure of any depth.
- The value attribute is kept separate. If value is present, the algorithm knows to stop and give an answer. If it's missing, the algorithm knows it must look at the feature and threshold to decide where to go next.

## Measuring disorder with entropy

Since we have already created the structure to hold the data (Nodes), now we need a way to decide how to split it. The goal of using a decision tree is to take a messy group of data and split it into groups that are "pure". For this exact reason we are going to use Shanon entropy to quantify this messiness.

In [2]:
import numpy as np

def calculate_entropy(y):
    """Calculate the Shannon entropy of a label array."""

    #1. We count how many times each class appears.
    counts = np.bincount(y)
    
    #2. We calculate the probability (pc) of each class.
    #pc = count of class / total number of samples
    probabilities = counts / len(y)
    
    #3. We apply the Shannon Entropy formula 
    # We use 'if p > 0' because to avoid log(0) which is undefined.
    entropy = -np.sum([p * np.log2(p) for p in probabilities if p > 0])
    
    return entropy

By calcualting this entropy we will get the following result:
- Zero entropy: If our node only contain one class, the entropy will be 0, meaning what we have is completely pure.
- High entropy: If the classes are evenly split (e.g 50% positive and 50% negative) we will get an entropy of 1 (in case of binary classification)

## Calculating information gain

Usin the entropy we now have the tool to measure the "disorder" of a single group. In order to calculate how much that disorder drops after we perform a spil, we need to calculate the information gain (ig). The main goal of such algorithm is to look at every possible feature and every possible value, and chose the one that results in the highest information gain.

#### Parent node vs. child node

To calculate the gain, we compare the entropy of the "Parent" node (before the split) to the weighted average entropy of the "Children" nodes (after the split)
- Start with Parent Entropy: How messy is the data right now?
- Calculate Child Entropy: If we split by "Feature X > 5", how messy are the two resulting groups?
- Weight the Children: A large child node is more important than a tiny one, so we multiply each child's entropy by the proportion of samples it contains
- Subtract: Gain = Parent Entropy - Weighted Child Entropy

In [3]:
def calculate_information_gain(y, left_idxs, right_idxs):
    #1. Entropy of the parent node
    parent_entropy = calculate_entropy(y)

    #2. Calculating weights for the children nodes
    n_total = len(y)
    n_left, n_right = len(left_idxs), len(right_idxs)

    #In case one of the children node is empty, we return 0 IG
    if n_left == 0 or n_right == 0:
        return 0

    #3. We calculate the entropy of the children nodes.
    #We get the labels (y) for the left and right splits using the provided indices.
    entropy_left = calculate_entropy(y[left_idxs])
    entropy_right = calculate_entropy(y[right_idxs])
    
    child_entropy = (n_left / n_total) * entropy_left + (n_right / n_total) * entropy_right

    #4. Finally, the information gain, whic is the reduction in entropy.
    ig = parent_entropy - child_entropy
    return ig

## Finding the best split

Now that we have a way to measure improvement (information gain), we need a mechanism to find the absolute best way to divide the data. To achieve this the algorithm must check every feature and every possible value (threshold) within those features to see which one reduces the "disorder" most.

In a decision tree, the split which is qualified as the "Best split" is the one that yields the largest reduction in impurity. To find it, the algorithm performs an exhaustive search:
- Iterate through features: The algorithm loops through every column in the dataset.
- Iterate through thresholds: For numerical features, it tests different values (e.g., $x \le 3$, $x \le 5$) as potential "splitting points"
- Evaluate: For every combination of (feature, threshold), it calculates the information gain.
- It saves only the combination that provides the highest reduction in entropy6.

In [4]:
def find_best_split(X, y):
    best_gain = -1
    split_idx, split_threshold = None, None
    
    #We check each feature (column) in the dataset
    n_features = X.shape[1]

    for feat_idx in range(n_features):
        #Extract the column of data for this feature
        X_column = X[:, feat_idx]
        
        #Get all unique values in this column to test as potential thresholds
        thresholds = np.unique(X_column)

        for threshold in thresholds:
            #1. Simulate the split based on the current threshold
            left_idxs = np.where(X_column <= threshold)[0]
            right_idxs = np.where(X_column > threshold)[0]

            #2. Calculate the Information Gain for this specific split
            gain = calculate_information_gain(y, left_idxs, right_idxs)

            #3. If this gain is the best we've seen so far we update our records.
            if gain > best_gain:
                best_gain = gain
                split_idx = feat_idx
                split_threshold = threshold

    return split_idx, split_threshold

## Recursive partioning

Now as we know how to find the best split, we need to apply this algorithm to grow our entire tree. Our algorithm starts at the Root Node with the entire dataset. It finds the best split, divides the data into two groups, and then treats those groups as "new" datasets to be split again. This continues until a Stopping Condition is met, which are typically:
- Pure node: All samples in the node belong to the same class.
- Maximum Depth: The tree has reached a user-defined limit of levels.
- Minimum Samples: There aren't enough samples left in the node to justify another split.

In [5]:
def grow_tree(X, y, depth=0, max_depth=10, min_samples_split=2):
    n_samples, n_features = X.shape
    n_labels = len(np.unique(y))

    #1. Check Stopping Conditions
    if (n_labels == 1 or depth >= max_depth or n_samples < min_samples_split):
        #The value is the most common class in this node (majority vote)
        leaf_value = np.argmax(np.bincount(y))
        return Node(value=leaf_value)

    #2. Find the best split.
    best_feat, best_thresh = find_best_split(X, y)
    
    #If no split improves the model, make it a leaf
    if best_feat is None:
        leaf_value = np.argmax(np.bincount(y))
        return Node(value=leaf_value)

    #3. Partition the Data
    left_idxs = np.where(X[:, best_feat] <= best_thresh)[0]
    right_idxs = np.where(X[:, best_feat] > best_thresh)[0]

    #4. Recursion: Grow the left and right child nodes
    left_child = grow_tree(X[left_idxs, :], y[left_idxs], depth + 1, max_depth)
    right_child = grow_tree(X[right_idxs, :], y[right_idxs], depth + 1, max_depth)

    #Return the current Decision Node with its children attached
    return Node(feature=best_feat, threshold=best_thresh, left=left_child, right=right_child)

## Making predictions

Now that we have our tree structure, the final step is to use it to predict the class of new, unseen data. This process involves traversing the tree from the root down to a leaf node. At each internal node, you apply the stored decision rule to determine whether to go left or right.

To make a prediction for a single data point:
- We start at the root: Here we look at the feature and threshold stored in the root node.
- Then we evaluate the tule: If the data point's value for that feature is less than or equal to the threshold, we follow the left branch, otherwise, follow the right branch.
- Continue this process at each internal node until you reach a leaf Node.
- Once you hit a leaf, the prediction is the value stored in that leaf (the majority class).

In [6]:
def predict_single(x, node):
    #If we are at a leaf, return the stored value
    if node.is_leaf_node():
        return node.value

    #Compare the feature value to the threshold
    if x[node.feature] <= node.threshold:
        #Go down the left branch
        return predict_single(x, node.left)
    else:
        # Go down the right branch
        return predict_single(x, node.right)

def predict(X, root_node):
    #Apply this traversal to every row in the input dataset X
    return np.array([predict_single(x, root_node) for x in X])

## Pruning

Once a tree is fully grown, it often becomes overly complex by memorizing specific noise in the training data. The porblem with this is that it leads to overfitting, meaning the model has zero training error but performs poorly on new, unseen data. The challenge is to find a balance where the tree is complex enough to capture patterns but simple enough to ignore noise.

#### Pre-pruning (Early stopping)
The easiest way to implement "pruning" is to stop the tree from growing too large in the first place. To achieve this we add constraints to the recursive function:
- Max depth: Stop splitting once the tree reaches a certain number of levels.
- Min samples per split: Only split a node if it contains a minimum number of samples.
- Min impurity decrease: Only allow a split if the Information Gain is above a certain threshold.

In [7]:
def grow_tree_with_pruning(X, y, depth=0, max_depth=5, min_samples=10, min_gain=0.01):
    n_samples, n_features = X.shape
    n_labels = len(np.unique(y))

    #We check if we should stop early
    if (n_labels == 1 or depth >= max_depth or n_samples < min_samples):
        leaf_value = np.argmax(np.bincount(y))
        return Node(value=leaf_value)

    #Then find the best split
    best_feat, best_thresh = find_best_split(X, y)

    #Calculate information gain for this split
    if best_feat is not None:
        left_idxs = np.where(X[:, best_feat] <= best_thresh)[0]
        right_idxs = np.where(X[:, best_feat] > best_thresh)[0]
        gain = calculate_information_gain(y, left_idxs, right_idxs)
        
        #Stop if the gain is too small
        if gain < min_gain:
            leaf_value = np.argmax(np.bincount(y))
            return Node(value=leaf_value)

        #Recursion if all checks pass
        left_child = grow_tree_with_pruning(X[left_idxs, :], y[left_idxs], depth + 1, max_depth, min_samples, min_gain)
        right_child = grow_tree_with_pruning(X[right_idxs, :], y[right_idxs], depth + 1, max_depth, min_samples, min_gain)
        
        return Node(feature=best_feat, threshold=best_thresh, left=left_child, right=right_child)

    return Node(value=np.argmax(np.bincount(y)))

## Testing the model on a real world example

In this example I am going to perform a credit risk assesment, meaning I would like to predict wheter the candidate for the loan will pay back the loan or not based on certain features.



Since this is a basic model I will check the most trivial features to asses the risk of each customer of the bank.
- Credit Score: Was the preson reliable in the past?
- Has debt: Are they already paying other debts?
- Income: Do they make enough many two cover a loan?
- Employement: Is their income stable?

When running the algorithm we can ask ourself, what is the most important question to ask first? Should we decide based on the credit score or the income of the customer? 
By using the decision tree model we operate based on information gain, this way it will decide based on the question which clears up "most" confusion.

In [8]:
#Let's create a simple dataset to test our decision tree implementation.
#Each row is a person: [credit score, has debt (0/1), income (k), years employed]
X_multi = np.array([
    [750, 0, 100, 10], #Person 1: high score, stable, high income -> approved
    [800, 0, 120, 15], #Person 2: perfect profile -> approved
    [650, 1, 90,  2],  #Person 3: good score, but debt and new job -> denied
    [500, 0, 40,  1],  #Person 4: low everything -> denied
    [710, 1, 150, 5],  #Person 5: has debt, but high income/stability -> approved
    [580, 1, 35,  1],  #Person 6: low score and low income -> denied
    [620, 0, 50,  8],  #Person 7: low score, but no debt and stable -> approved
])

#1 = approved, 0 = denied
#based on the features above we can label them accordingly
y_multi = np.array([1, 1, 0, 0, 1, 0, 1])

#When we grow the tree, I set the 'max_depth=3' which allows the tree to ask up to 3 questions.
#before making a final decision.
multi_tree = grow_tree(X_multi, y_multi, max_depth=3)

Now that we have trained our model, we can predict wheter new customers should be approved for taking out loan or not.

To test this I have created two completelly different profiles:

1. Applicant A (the safe choice):
- High credit score: 780
- No debt: 0
- High income: 110
- Stability: 12

2. Applicant B (the risky choice):
- Below average credit score: 590
- Has debt: 1
- Modest income: 42
- Low stability: 1

In [10]:
#Now let's test our decision tree with two hypothetical applicants.
applicant_a = np.array([780, 0, 110, 12])
applicant_b = np.array([590, 1, 42, 1])

#We use the previously defined 'predict_single' function to classify each applicant.
result_a = predict_single(applicant_a, multi_tree)
result_b = predict_single(applicant_b, multi_tree)

print(f"Applicant A (The Professional): {'APPROVED' if result_a == 1 else 'DENIED'}")
print(f"Applicant B (The High-Risk): {'APPROVED' if result_b == 1 else 'DENIED'}")

Applicant A (The Professional): APPROVED
Applicant B (The High-Risk): DENIED
